In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/clean_data.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Shape: (2052, 11)

Columns: ['status', 'floor', 'transaction', 'bathroom', 'Rate_per_sqft', 'bedroom', 'carpet_area_sqft', 'area', 'Price_Lakhs', 'area_median_ppsf', 'ppsf_vs_area']

First 3 rows:


,status,floor,transaction,bathroom,Rate_per_sqft,bedroom,carpet_area_sqft,area,Price_Lakhs,area_median_ppsf,ppsf_vs_area
0,Ready to Move,2,Resale,5.0,21250.0,4,3000.0,New Delhi,850.0,8796.0,2.415871
1,Ready to Move,0,New Property,4.0,9815.0,4,1500.0,Gurgaon,265.0,9259.0,1.060050
2,Ready to Move,1,New Property,2.0,10083.0,3,1100.0,New Delhi,121.0,8796.0,1.146317


In [4]:
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# Columns that contain text
categorical_cols = ['status', 'transaction', 'area']

# Create a dictionary to store encoders
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

status: {'Ready to Move': 0, 'Under Construction': 1}
transaction: {'New Property': 0, 'Other': 1, 'Resale': 2}
area: {'Faridabad': 0, 'Ghaziabad': 1, 'Greater Noida': 2, 'Gurgaon': 3, 'New Delhi': 4, 'Noida': 5}


In [6]:
# Save encoders for use in the app later
os.makedirs('../model', exist_ok=True)
joblib.dump(encoders, '../model/encoders.pkl')
print("Encoders saved!")

# Check data now
print("\nData after encoding:")
df.head(3)

Encoders saved!

Data after encoding:


,status,floor,transaction,bathroom,Rate_per_sqft,bedroom,carpet_area_sqft,area,Price_Lakhs,area_median_ppsf,ppsf_vs_area
0,0,2,2,5.0,21250.0,4,3000.0,4,850.0,8796.0,2.415871
1,0,0,0,4.0,9815.0,4,1500.0,3,265.0,9259.0,1.060050
2,0,1,0,2.0,10083.0,3,1100.0,4,121.0,8796.0,1.146317


In [8]:
from sklearn.model_selection import train_test_split

# X = everything we use to predict
# y = what we're predicting (price)
X = df.drop(columns=['Price_Lakhs'])
y = df['Price_Lakhs']

# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nFeatures used:", X.columns.tolist())

Training samples: 1641
Testing samples: 411

Features used: ['status', 'floor', 'transaction', 'bathroom', 'Rate_per_sqft', 'bedroom', 'carpet_area_sqft', 'area', 'area_median_ppsf', 'ppsf_vs_area']


In [10]:
# Model 1: Linear Regression
with mlflow.start_run(run_name="linear_regression"):
    model_lr = LinearRegression()
    model_lr.fit(X_train, y_train)
    preds_lr = model_lr.predict(X_test)
    
    r2_lr = r2_score(y_test, preds_lr)
    mae_lr = mean_absolute_error(y_test, preds_lr)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("r2_score", r2_lr)
    mlflow.log_metric("mae_lakhs", mae_lr)
    
    print(f"Linear Regression → R²: {r2_lr:.3f}, MAE: ₹{mae_lr:.1f}L")

# Model 2: Random Forest
with mlflow.start_run(run_name="random_forest"):
    model_rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42
    )
    model_rf.fit(X_train, y_train)
    preds_rf = model_rf.predict(X_test)
    
    r2_rf = r2_score(y_test, preds_rf)
    mae_rf = mean_absolute_error(y_test, preds_rf)
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 15)
    mlflow.log_metric("r2_score", r2_rf)
    mlflow.log_metric("mae_lakhs", mae_rf)
    
    print(f"Random Forest    → R²: {r2_rf:.3f}, MAE: ₹{mae_rf:.1f}L")

# Model 3: XGBoost
with mlflow.start_run(run_name="xgboost"):
    model_xgb = XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
    model_xgb.fit(X_train, y_train)
    preds_xgb = model_xgb.predict(X_test)
    
    r2_xgb = r2_score(y_test, preds_xgb)
    mae_xgb = mean_absolute_error(y_test, preds_xgb)
    
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("r2_score", r2_xgb)
    mlflow.log_metric("mae_lakhs", mae_xgb)
    
    print(f"XGBoost          → R²: {r2_xgb:.3f}, MAE: ₹{mae_xgb:.1f}L")

print("\n✅ All models trained!")
print("\nWINNER:")
scores = {"Linear Regression": r2_lr, "Random Forest": r2_rf, "XGBoost": r2_xgb}
winner = max(scores, key=scores.get)
print(f"{winner} with R²: {scores[winner]:.3f}")

NameError: name 'mlflow' is not defined

In [12]:
import subprocess
subprocess.run(['pip', 'install', 'mlflow', 'xgboost'], capture_output=True)

import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

mlflow.set_experiment("predict-home")
print("All imports done. Ready to train!")


ImportError: cannot import name 'Sentinel' from 'typing_extensions' (/opt/anaconda3/lib/python3.12/site-packages/typing_extensions.py)

In [3]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

mlflow.set_experiment("predict-home")
print("All imports done. Ready to train!")

All imports done. Ready to train!


In [5]:
# Model 1: Linear Regression
with mlflow.start_run(run_name="linear_regression"):
    model_lr = LinearRegression()
    model_lr.fit(X_train, y_train)
    preds_lr = model_lr.predict(X_test)
    
    r2_lr = r2_score(y_test, preds_lr)
    mae_lr = mean_absolute_error(y_test, preds_lr)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("r2_score", r2_lr)
    mlflow.log_metric("mae_lakhs", mae_lr)
    
    print(f"Linear Regression → R²: {r2_lr:.3f}, MAE: ₹{mae_lr:.1f}L")

# Model 2: Random Forest
with mlflow.start_run(run_name="random_forest"):
    model_rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42
    )
    model_rf.fit(X_train, y_train)
    preds_rf = model_rf.predict(X_test)
    
    r2_rf = r2_score(y_test, preds_rf)
    mae_rf = mean_absolute_error(y_test, preds_rf)
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 15)
    mlflow.log_metric("r2_score", r2_rf)
    mlflow.log_metric("mae_lakhs", mae_rf)
    
    print(f"Random Forest    → R²: {r2_rf:.3f}, MAE: ₹{mae_rf:.1f}L")

# Model 3: XGBoost
with mlflow.start_run(run_name="xgboost"):
    model_xgb = XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
    model_xgb.fit(X_train, y_train)
    preds_xgb = model_xgb.predict(X_test)
    
    r2_xgb = r2_score(y_test, preds_xgb)
    mae_xgb = mean_absolute_error(y_test, preds_xgb)
    
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("r2_score", r2_xgb)
    mlflow.log_metric("mae_lakhs", mae_xgb)
    
    print(f"XGBoost          → R²: {r2_xgb:.3f}, MAE: ₹{mae_xgb:.1f}L")

print("\n✅ All models trained!")
print("\nWINNER:")
scores = {"Linear Regression": r2_lr, "Random Forest": r2_rf, "XGBoost": r2_xgb}
winner = max(scores, key=scores.get)
print(f"{winner} with R²: {scores[winner]:.3f}")

NameError: name 'X_train' is not defined

In [7]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# Load clean data
df = pd.read_csv('../data/clean_data.csv')

# Encode categorical columns
categorical_cols = ['status', 'transaction', 'area']
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Split features and target
X = df.drop(columns=['Price_Lakhs'])
y = df['Price_Lakhs']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# MLflow setup
mlflow.set_experiment("predict-home")

print("Everything loaded. Ready to train!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Everything loaded. Ready to train!
Training samples: 1641
Testing samples: 411


In [9]:
# Model 1: Linear Regression
with mlflow.start_run(run_name="linear_regression"):
    model_lr = LinearRegression()
    model_lr.fit(X_train, y_train)
    preds_lr = model_lr.predict(X_test)
    
    r2_lr = r2_score(y_test, preds_lr)
    mae_lr = mean_absolute_error(y_test, preds_lr)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("r2_score", r2_lr)
    mlflow.log_metric("mae_lakhs", mae_lr)
    
    print(f"Linear Regression → R²: {r2_lr:.3f}, MAE: ₹{mae_lr:.1f}L")

# Model 2: Random Forest
with mlflow.start_run(run_name="random_forest"):
    model_rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42
    )
    model_rf.fit(X_train, y_train)
    preds_rf = model_rf.predict(X_test)
    
    r2_rf = r2_score(y_test, preds_rf)
    mae_rf = mean_absolute_error(y_test, preds_rf)
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 15)
    mlflow.log_metric("r2_score", r2_rf)
    mlflow.log_metric("mae_lakhs", mae_rf)
    
    print(f"Random Forest    → R²: {r2_rf:.3f}, MAE: ₹{mae_rf:.1f}L")

# Model 3: XGBoost
with mlflow.start_run(run_name="xgboost"):
    model_xgb = XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
    model_xgb.fit(X_train, y_train)
    preds_xgb = model_xgb.predict(X_test)
    
    r2_xgb = r2_score(y_test, preds_xgb)
    mae_xgb = mean_absolute_error(y_test, preds_xgb)
    
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("r2_score", r2_xgb)
    mlflow.log_metric("mae_lakhs", mae_xgb)
    
    print(f"XGBoost          → R²: {r2_xgb:.3f}, MAE: ₹{mae_xgb:.1f}L")

print("\n✅ All models trained!")
print("\nWINNER:")
scores = {"Linear Regression": r2_lr, "Random Forest": r2_rf, "XGBoost": r2_xgb}
winner = max(scores, key=scores.get)
print(f"{winner} with R²: {scores[winner]:.3f}")

Linear Regression → R²: 0.415, MAE: ₹93.3L
Random Forest    → R²: 0.504, MAE: ₹74.7L
XGBoost          → R²: 0.473, MAE: ₹78.2L

✅ All models trained!

WINNER:
Random Forest with R²: 0.504


In [11]:
# Check correlation of each feature with price
correlations = X.corrwith(y).sort_values(ascending=False)
print("Feature correlations with Price:")
print(correlations)


Feature correlations with Price:
bathroom            0.545717
bedroom             0.530729
area_median_ppsf    0.280893
Rate_per_sqft       0.224535
ppsf_vs_area        0.179613
area                0.164215
carpet_area_sqft    0.103041
status              0.025444
floor              -0.000352
transaction        -0.092112
dtype: float64


In [13]:
# Check how many rows have the median value for carpet_area_sqft
original_df = pd.read_csv('../data/clean_data.csv')
median_val = original_df['carpet_area_sqft'].median()
filled_count = (original_df['carpet_area_sqft'] == median_val).sum()
print(f"Median value: {median_val}")
print(f"Rows with median value: {filled_count} out of {len(original_df)}")
print(f"Percentage: {filled_count/len(original_df)*100:.1f}%")

Median value: 1150.0
Rows with median value: 1094 out of 2052
Percentage: 53.3%


In [15]:
# Drop carpet_area_sqft
X = X.drop(columns=['carpet_area_sqft'])

# Retrain test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Features now:", X.columns.tolist())
print("Training samples:", len(X_train))

Features now: ['status', 'floor', 'transaction', 'bathroom', 'Rate_per_sqft', 'bedroom', 'area', 'area_median_ppsf', 'ppsf_vs_area']
Training samples: 1641


In [17]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# Load clean data
df = pd.read_csv('../data/clean_data.csv')

# Drop carpet_area_sqft (53% was fake median values)
df = df.drop(columns=['carpet_area_sqft'])

# Encode categorical columns
categorical_cols = ['status', 'transaction', 'area']
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Save encoders
os.makedirs('../model', exist_ok=True)
joblib.dump(encoders, '../model/encoders.pkl')

# Split features and target
X = df.drop(columns=['Price_Lakhs'])
y = df['Price_Lakhs']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("predict-home")

print("Ready! Features:", X.columns.tolist())
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Ready! Features: ['status', 'floor', 'transaction', 'bathroom', 'Rate_per_sqft', 'bedroom', 'area', 'area_median_ppsf', 'ppsf_vs_area']
Training samples: 1641
Testing samples: 411


In [19]:
# Model 1: Linear Regression
with mlflow.start_run(run_name="linear_regression_v2"):
    model_lr = LinearRegression()
    model_lr.fit(X_train, y_train)
    preds_lr = model_lr.predict(X_test)
    r2_lr = r2_score(y_test, preds_lr)
    mae_lr = mean_absolute_error(y_test, preds_lr)
    mlflow.log_param("model_type", "LinearRegression_v2")
    mlflow.log_metric("r2_score", r2_lr)
    mlflow.log_metric("mae_lakhs", mae_lr)
    print(f"Linear Regression → R²: {r2_lr:.3f}, MAE: ₹{mae_lr:.1f}L")

# Model 2: Random Forest
with mlflow.start_run(run_name="random_forest_v2"):
    model_rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
    model_rf.fit(X_train, y_train)
    preds_rf = model_rf.predict(X_test)
    r2_rf = r2_score(y_test, preds_rf)
    mae_rf = mean_absolute_error(y_test, preds_rf)
    mlflow.log_param("model_type", "RandomForest_v2")
    mlflow.log_metric("r2_score", r2_rf)
    mlflow.log_metric("mae_lakhs", mae_rf)
    print(f"Random Forest    → R²: {r2_rf:.3f}, MAE: ₹{mae_rf:.1f}L")

# Model 3: XGBoost
with mlflow.start_run(run_name="xgboost_v2"):
    model_xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, verbosity=0)
    model_xgb.fit(X_train, y_train)
    preds_xgb = model_xgb.predict(X_test)
    r2_xgb = r2_score(y_test, preds_xgb)
    mae_xgb = mean_absolute_error(y_test, preds_xgb)
    mlflow.log_param("model_type", "XGBoost_v2")
    mlflow.log_metric("r2_score", r2_xgb)
    mlflow.log_metric("mae_lakhs", mae_xgb)
    print(f"XGBoost          → R²: {r2_xgb:.3f}, MAE: ₹{mae_xgb:.1f}L")

print("\n✅ All models retrained!")
scores = {"Linear Regression": r2_lr, "Random Forest": r2_rf, "XGBoost": r2_xgb}
winner = max(scores, key=scores.get)
print(f"\nWINNER: {winner} with R²: {scores[winner]:.3f}")

Linear Regression → R²: 0.416, MAE: ₹93.3L
Random Forest    → R²: 0.519, MAE: ₹74.4L
XGBoost          → R²: 0.502, MAE: ₹74.6L

✅ All models retrained!

WINNER: Random Forest with R²: 0.519


In [21]:
# Apply log transform to price
import numpy as np

y_log = np.log1p(y)  # log1p = log(1+x), handles zeros safely

# Retrain with log price
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

_, _, y_train_orig, y_test_orig = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Random Forest on log price
with mlflow.start_run(run_name="rf_log_transform"):
    model_rf_log = RandomForestRegressor(
        n_estimators=100, max_depth=15, random_state=42
    )
    model_rf_log.fit(X_train, y_train_log)
    preds_log = model_rf_log.predict(X_test)
    
    # Convert back to original scale
    preds_original = np.expm1(preds_log)
    
    r2_log = r2_score(y_test_orig, preds_original)
    mae_log = mean_absolute_error(y_test_orig, preds_original)
    
    mlflow.log_param("model_type", "RF_LogTransform")
    mlflow.log_metric("r2_score", r2_log)
    mlflow.log_metric("mae_lakhs", mae_log)
    
    print(f"RF + Log Transform → R²: {r2_log:.3f}, MAE: ₹{mae_log:.1f}L")

# Train XGBoost on log price
with mlflow.start_run(run_name="xgb_log_transform"):
    model_xgb_log = XGBRegressor(
        n_estimators=200, learning_rate=0.1,
        max_depth=6, random_state=42, verbosity=0
    )
    model_xgb_log.fit(X_train, y_train_log)
    preds_log = model_xgb_log.predict(X_test)
    
    preds_original = np.expm1(preds_log)
    
    r2_xgb_log = r2_score(y_test_orig, preds_original)
    mae_xgb_log = mean_absolute_error(y_test_orig, preds_original)
    
    mlflow.log_param("model_type", "XGB_LogTransform")
    mlflow.log_metric("r2_score", r2_xgb_log)
    mlflow.log_metric("mae_lakhs", mae_xgb_log)
    
    print(f"XGB + Log Transform → R²: {r2_xgb_log:.3f}, MAE: ₹{mae_xgb_log:.1f}L")

RF + Log Transform → R²: 0.461, MAE: ₹73.9L
XGB + Log Transform → R²: 0.476, MAE: ₹70.2L
